# JevBench CLI: scoring a trained decider on the public items

A trained decider (an arm's final weights from `05_jepa_point.ipynb`, or a later run) is served on this kernel at `POST /v1/systemone`, the request and answer format of JevBench's `typesafe` adapter. JevBench's own CLI, at a pinned commit, then runs all 231 public items against it and summarises them.

- **Server:** one GPU, one request at a time, the model code of 05 unchanged. A state too long for `EVAL_MAX_LEN` is cut, not refused.
- **Answers:** `noul` → P(yes); `choice` → the top option and every option's probability; `score` → every level's probability.
- **The CLI's numbers:** accuracy, Brier, ECE, ordinal MAE, latency, schema validity.
- **Intelligence (our estimate):** the board's official scorer is not public. We apply its published rules to the per-item results: chance-corrected competence per request type (choice against a random option, noul against 50% with P(yes) inside (0.2, 0.8) counted as an abstention and so wrong, score against always guessing the middle level), the mean over the types present per tier, then tiers weighted easy 0.1, standard 0.2, hard 0.4 (the public `original` items are taken as standard; there is no judge tier in the public set). Shown as the model answers, and with every noul answer made decisive (the same top answer, P(yes) moved to 0.8 or 0.2), which some board entries do.

Public items only: this is not a board number (the board adds a sealed set, the judge tier and very long items).

## Parameters

In [ ]:
import os

NAME = "C2-base"                                    # the run's label
ENCODER = "answerdotai/ModernBERT-base"
WEIGHTS = "/kaggle/working/ckpt05/C2_final.pt"     # an arm's final weights
POINTER = True                                     # C2 arms: the pointer and its EMA target
READOUT = "both"                                   # head | point | both (C2's decision)
JEVBENCH_COMMIT = "609a664e81f46e03731600bd7bd9f1b7a61257c5"
JEVBENCH_DIR = "/kaggle/working/jevbench"          # the CLI's code
RUNS_DIR = "/kaggle/working/jevbench_runs"         # results, ledger, raw answers (outside the repo)
PORT = 8000
EVAL_MAX_LEN = 8192                                # longer states are cut, not refused
OPTION_MAX_TOKENS = 64
MAX_OPTIONS = 26
TAU_START = 0.1                                    # replaced by the trained value on loading
NOUL_DECISIVE = 0.8                                # P(yes) inside (0.2, 0.8) is an abstention
TIERS = {"easy": 0.1, "original": 0.2, "hard": 0.4}   # public file -> the board's tier weight

## Packages and JevBench at the pinned commit

The CLI needs only the standard library; it runs from its own folder.

In [ ]:
!pip install -q "transformers>=4.48"

import copy, json, math, statistics, subprocess, sys, threading, time
from http.server import BaseHTTPRequestHandler, HTTPServer

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForMaskedLM, AutoTokenizer

if not os.path.isdir(JEVBENCH_DIR):
    subprocess.run(["git", "clone", "-q", "https://github.com/fstandhartinger/jevbench.git",
                    JEVBENCH_DIR], check=True)
subprocess.run(["git", "-C", JEVBENCH_DIR, "fetch", "-q", "origin"], check=False)
subprocess.run(["git", "-C", JEVBENCH_DIR, "checkout", "-q", JEVBENCH_COMMIT], check=True)
head = subprocess.run(["git", "-C", JEVBENCH_DIR, "rev-parse", "HEAD"],
                      capture_output=True, text=True, check=True).stdout.strip()
assert head == JEVBENCH_COMMIT, head
TASK_FILES = {tier: f"{JEVBENCH_DIR}/datasets/public/{tier}.jsonl" for tier in TIERS}
print(f"JevBench at {head[:7]} | {sum(1 for f in TASK_FILES.values() for _ in open(f))} public items")

## Device and log

The server uses one GPU and answers one request at a time.

In [ ]:
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"
BF16_HW = AMP and torch.cuda.get_device_capability(0)[0] >= 8
AMP_DTYPE = torch.bfloat16 if BF16_HW else torch.float16


def amp():
    return torch.autocast("cuda", dtype=AMP_DTYPE, enabled=AMP)


def spread(module):
    return module                                  # serving: one GPU


def log(message):
    print(f"[{time.strftime('%H:%M:%S')}] {message}", flush=True)


log(f"torch {torch.__version__} | {torch.cuda.get_device_name() if AMP else 'CPU'} | {NAME} | "
    f"{ENCODER} | {WEIGHTS}")

## The model code of 05 (unchanged)

In [ ]:
def state_text(state):
    return state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)


def option_texts(q):
    """-> (option names, option texts) for one System One question."""
    kind, crit = q["type"], q.get("criteria")
    if kind == "noul":
        crit = crit or {}
        return ["true", "false"], [crit.get("true") or "", crit.get("false") or ""]
    if kind == "choice":
        names = list(crit)
        return names, [f"{n} - {d}" if d else n for n, d in crit.items()]
    return [str(i) for i in range(len(crit))], [str(level) for level in crit]

In [ ]:
tok = AutoTokenizer.from_pretrained(ENCODER)
CLS, SEP, PAD, MASK = tok.cls_token_id, tok.sep_token_id, tok.pad_token_id, tok.mask_token_id
LABELS = {"choice": [chr(65 + i) for i in range(MAX_OPTIONS)],
          "noul": ["yes", "no"],
          "score": [str(i) for i in range(10)]}


def ids(text):
    return tok(text, add_special_tokens=False)["input_ids"]


def one_token(label):
    found = ids(" " + label)
    assert len(found) == 1, f"label {label!r} is {len(found)} tokens"
    return found[0]


LABEL_IDS = {kind: [one_token(lab) for lab in labs] for kind, labs in LABELS.items()}
HEAD_TOKENS = LABEL_IDS["choice"] + LABEL_IDS["noul"] + LABEL_IDS["score"]   # the 38 head outputs
HEAD_SLOTS = {"choice": list(range(26)), "noul": [26, 27], "score": list(range(28, 38))}
for labs in LABELS.values():
    for lab in labs:
        assert ids(f"{lab}: x")[:len(ids(lab))] == ids(lab), f"label {lab!r} merges with ':'"
print({k: v[:4] for k, v in LABEL_IDS.items()}, "| head outputs", len(HEAD_TOKENS))

In [ ]:
def sequence(item, max_len):
    """-> (token ids with [MASK], mask position, label positions, state was cut)."""
    labels = LABELS[item["type"]]
    tail, at = [], []
    for label, text in zip(labels, item["texts"]):
        line = f"{label}: {text}".strip() if text else label
        at.append(len(tail) + len(ids(label)) - 1)        # the label's (last) token
        tail += ids(line)[:OPTION_MAX_TOKENS] + [SEP]
    tail += ids("Answer:")
    q = ids(item["question"])[:256]
    s = ids(item["state"])
    room = max_len - len(q) - len(tail) - 5
    cut = len(s) > room
    s = s[:max(room, 0)]
    head = [CLS] + q + [SEP] + s + [SEP]
    seq = head + tail
    pos = len(seq)
    seq += [MASK, SEP]
    return seq, pos, [len(head) + a for a in at], cut


def batch_tensors(items, max_len):
    rows = [sequence(it, max_len) for it in items]
    width = max(len(r[0]) for r in rows)
    k = max(len(r[2]) for r in rows)
    input_ids, att, pos, label_pos = [], [], [], []
    for seq, p, at, _ in rows:
        input_ids.append(seq + [PAD] * (width - len(seq)))
        att.append([1] * len(seq) + [0] * (width - len(seq)))
        pos.append(p)
        label_pos.append(at + [0] * (k - len(at)))        # padding points at [CLS]; masked out
    as_t = lambda x: torch.tensor(x, device=DEVICE)
    return as_t(input_ids), as_t(att), as_t(pos), as_t(label_pos), sum(r[3] for r in rows)

In [ ]:
class EncoderAt(nn.Module):
    """An encoder's vectors at given positions: one per row ([rows]) or several ([rows, k]);
    split across GPUs by spread()."""

    def __init__(self, encoder):
        super().__init__()
        self.enc = encoder

    def forward(self, input_ids, att, pos):
        with amp():                                   # each GPU's copy runs in mixed precision
            h = self.enc(input_ids=input_ids, attention_mask=att).last_hidden_state
        rows = torch.arange(len(pos), device=h.device)
        return h[rows, pos] if pos.dim() == 1 else h[rows[:, None], pos]


class MaskedDecider(nn.Module):
    def __init__(self):
        super().__init__()
        self.mlm = AutoModelForMaskedLM.from_pretrained(ENCODER, attn_implementation="sdpa")
        hidden = self.mlm.config.hidden_size
        for p in self.mlm.model.embeddings.parameters():
            p.requires_grad = False
        self.predictor = nn.Sequential(nn.Linear(hidden, 2 * hidden), nn.GELU(),
                                       nn.LayerNorm(2 * hidden), nn.Linear(2 * hidden, hidden))
        self.label_head = nn.Linear(hidden, len(HEAD_TOKENS))
        with torch.no_grad():                         # start from the pretrained output rows
            self.label_head.weight.copy_(self.mlm.decoder.weight[HEAD_TOKENS])
            if self.mlm.decoder.bias is not None:
                self.label_head.bias.copy_(self.mlm.decoder.bias[HEAD_TOKENS])
        self.log_tau = nn.Parameter(torch.tensor(math.log(TAU_START)))
        self.target = copy.deepcopy(self.mlm.model).requires_grad_(False)
        self.at_mask = spread(EncoderAt(self.mlm.model))
        self.target_at = spread(EncoderAt(self.target))

    def mask_vectors(self, input_ids, att, pos):
        return self.at_mask(input_ids, att, pos)

    def label_logits(self, vectors, items):
        """The 38-way head at [MASK]; each question keeps its own labels -> [items, options]."""
        all38 = self.label_head(self.mlm.head(vectors)).float()
        width = max(len(it["names"]) for it in items)
        out = torch.full((len(items), width), -1e4, device=vectors.device)
        for i, it in enumerate(items):
            k = len(it["names"])
            out[i, :k] = all38[i, HEAD_SLOTS[it["type"]][:k]]
        return out

    def tau(self):
        return self.log_tau.exp().clamp(min=0.01)

    def train(self, mode=True):
        """Train the online parts; the EMA target encoder always stays in eval mode."""
        super().train(mode)
        self.target.eval()
        return self

    @torch.no_grad()
    def target_vectors(self, input_ids, att, label_pos):
        return self.target_at(input_ids, att, label_pos)

In [ ]:
def forward(model, items, max_len, pointer):
    input_ids, att, pos, label_pos, cut = batch_tensors(items, max_len)
    v = model.mask_vectors(input_ids, att, pos)
    scores, extra = {"head": model.label_logits(v, items)}, {}
    if pointer:
        targets = model.target_vectors(input_ids, att, label_pos).float()   # [items, k, hidden]
        z = (v + model.predictor(v)).float()
        cos = F.cosine_similarity(z.unsqueeze(1), targets, dim=-1)
        k = torch.tensor([len(it["names"]) for it in items], device=cos.device)
        valid = torch.arange(cos.shape[1], device=cos.device)[None] < k[:, None]
        scores["point"] = (cos / model.tau()).masked_fill(~valid, -1e4)
        extra = {"z": z, "targets": targets, "cos": cos, "valid": valid}
    return scores, extra, cut


def readouts(scores):
    """Probabilities per readout: the head, the pointer, and both averaged."""
    probs = {name: F.softmax(s, dim=1) for name, s in scores.items()}
    if "point" in probs:
        probs["both"] = (probs["head"] + probs["point"]) / 2
    return probs

## The trained weights

Saved from a two-GPU run, so the keys of the GPU-split modules carry `.module.`; it is removed to load them on one GPU. Every key must match.

In [ ]:
model = MaskedDecider().to(DEVICE)
saved = torch.load(WEIGHTS, map_location="cpu", weights_only=False)
saved = {k.replace(".module.", "."): v for k, v in saved.items()}
missing, unexpected = model.load_state_dict(saved, strict=False)
assert not missing and not unexpected, (missing[:5], unexpected[:5])
model.eval()
del saved
log(f"loaded {WEIGHTS} | tau {model.tau().item():.3f}")

## The answer to one request

`noul` options are the criteria's `true` / `false` texts, so the first probability is P(yes); `score` levels are their indices; `choice` options are the criteria's keys. Probabilities are renormalised in double precision so they sum to 1.

In [ ]:
@torch.no_grad()
def decide(state, q):
    names, texts = option_texts(q)
    if len(names) > MAX_OPTIONS:
        raise ValueError(f"{len(names)} options, at most {MAX_OPTIONS}")
    item = {"type": q["type"], "question": q.get("instructions") or "", "state": state_text(state),
            "names": names, "texts": texts, "gold": [1 / len(names)] * len(names)}
    with amp():
        scores, _, _ = forward(model, [item], EVAL_MAX_LEN, POINTER)
    p = readouts(scores)[READOUT][0, :len(names)].double()
    return names, (p / p.sum()).tolist(), len(sequence(item, EVAL_MAX_LEN)[0])


def answer(body):
    q = body["questions"]["decision"]
    names, p, n_tokens = decide(body["state"], q)
    if q["type"] == "noul":
        out = {"type": "noul", "noul": p[0]}
    elif q["type"] == "choice":
        out = {"type": "choice", "choice": names[max(range(len(p)), key=p.__getitem__)],
               "probabilities": dict(zip(names, p))}
    else:
        out = {"type": "score", "score": sum(i * x for i, x in enumerate(p)),
               "probabilities": {str(i): x for i, x in enumerate(p)}}
    return {"model": NAME, "answers": {"decision": out},
            "usage": {"input_tokens": n_tokens, "output_tokens": 0}}

## The server

`POST /v1/systemone` on `127.0.0.1:PORT`, in a background thread of this kernel; a bad request gets HTTP 400 with the error. Running the cell again restarts it.

In [ ]:
class Handler(BaseHTTPRequestHandler):
    def do_POST(self):
        if self.path != "/v1/systemone":
            return self.reply(404, {"error": "not found"})
        try:
            body = json.loads(self.rfile.read(int(self.headers["Content-Length"])))
            self.reply(200, answer(body))
        except Exception as e:
            self.reply(400, {"error": f"{type(e).__name__}: {e}"})

    def reply(self, status, obj):
        data = json.dumps(obj).encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(data)))
        self.end_headers()
        self.wfile.write(data)

    def log_message(self, *args):
        pass


old_server = globals().get("server")
if old_server is not None:                         # the cell run again: stop the old one first
    old_server.shutdown()
    old_server.server_close()
server = HTTPServer(("127.0.0.1", PORT), Handler)
threading.Thread(target=server.serve_forever, daemon=True).start()
log(f"serving {NAME} at http://127.0.0.1:{PORT}/v1/systemone")

## A check through JevBench's own adapter and scorer

One public item of each type goes through the `typesafe` adapter and `score_task`, exactly as the CLI will send and score it.

In [ ]:
sys.path.insert(0, JEVBENCH_DIR)
from jevbench.adapters.typesafe import TypeSafeAdapter
from jevbench.scoring import score_task
from jevbench.tasks import load_jsonl

tasks = {tier: load_jsonl(path) for tier, path in TASK_FILES.items()}
tier_of = {t.id: tier for tier, ts in tasks.items() for t in ts}
by_id = {t.id: t for ts in tasks.values() for t in ts}
adapter = TypeSafeAdapter(endpoint=f"http://127.0.0.1:{PORT}", model=NAME, key_env="")
for kind in ("noul", "choice", "score"):
    t = next(t for t in by_id.values() if t.question["type"] == kind)
    r = adapter.run(t)
    assert r.ok, r.error
    s = score_task(r.probs, t)
    assert s["valid"], s
    log(f"{kind}: {t.id} | valid | predicted {s['predicted']} | expected {t.expected} | "
        f"{r.latency_s * 1000:.0f} ms")

## The CLI run

All public items, in one serial run, into a new folder under `RUNS_DIR` (the runner refuses to write inside its own repo). No billable account: the tariff is 0 and the cost basis says why.

In [ ]:
run_dir = f"{RUNS_DIR}/{NAME}-{time.strftime('%Y%m%d-%H%M%S')}"
os.makedirs(run_dir)
all_tasks = ",".join(TASK_FILES.values())
cmd = [sys.executable, "-m", "jevbench.cli", "run", "--tasks", all_tasks, "--adapter", "typesafe",
       "--endpoint", f"http://127.0.0.1:{PORT}", "--model", NAME, "--key-env", "",
       "--results", f"{run_dir}/results.jsonl", "--ledger", f"{run_dir}/ledger.jsonl",
       "--raw-dir", f"{run_dir}/raw", "--reserve-usd", "0", "--price-in-per-m", "0",
       "--price-out-per-m", "0", "--cost-basis", "local_gpu_no_billable_account",
       "--run-label", NAME, "--manifest", f"{run_dir}/manifest.json"]
t0 = time.perf_counter()
done = subprocess.run(cmd, cwd=JEVBENCH_DIR, capture_output=True, text=True)
print(done.stdout[-2000:], done.stderr[-2000:])
log(f"CLI run: exit {done.returncode} in {(time.perf_counter() - t0) / 60:.1f} min -> {run_dir}")

## The CLI's summary

Overall and per tier (each tier's results summarised against its own public file).

In [ ]:
def cli_summary(task_files, results):
    out = subprocess.run([sys.executable, "-m", "jevbench.cli", "summarize", "--tasks", task_files,
                          "--results", results],
                         cwd=JEVBENCH_DIR, capture_output=True, text=True, check=True)
    return json.loads(out.stdout)


summaries = {"all": cli_summary(all_tasks, f"{run_dir}/results.jsonl")}
lines = open(f"{run_dir}/results.jsonl").read().splitlines()
for tier, path in TASK_FILES.items():
    wanted = {json.loads(line)["id"] for line in open(path)}
    part = f"{run_dir}/results-{tier}.jsonl"                    # the summariser wants one file's items
    with open(part, "w") as fh:
        fh.writelines(line + "\n" for line in lines if json.loads(line)["task_id"] in wanted)
    summaries[tier] = cli_summary(path, part)
json.dump(summaries, open(f"{run_dir}/summary.json", "w"), indent=1)
print(f"{'':9s}{'n':>5s} {'acc':>7s} {'brier':>7s} {'ece':>7s} {'ord MAE':>8s} {'valid':>7s} "
      f"{'p50 ms':>7s} {'p95 ms':>7s}")
for name, s in summaries.items():
    fmt = lambda v, d=3: "-" if v is None else f"{v:.{d}f}"
    print(f"{name:9s}{s['n_scorable']:>5d} {fmt(s['accuracy']):>7s} {fmt(s['brier_mean']):>7s} "
          f"{fmt(s['ece']['ece']):>7s} {fmt(s['ordinal_mae']):>8s} {fmt(s['schema_validity']):>7s} "
          f"{s['latency']['p50_s'] * 1000:>7.0f} {s['latency']['p95_s'] * 1000:>7.0f}")
print(f"failed or invalid answers: {sum(1 for _ in open(f'{run_dir}/results.jsonl')) - summaries['all']['n_valid']}"
      f" | macro accuracy over families {summaries['all']['macro_accuracy']:.3f}")

## Intelligence, our estimate from the published rules

Per tier and request type, chance-corrected competence (0 = baseline, 100 = perfect, can be negative): choice against a random option, noul against 50% with abstentions wrong, score against the error of always guessing the middle level (on the expected level). Per tier the mean over the types present, clipped to 0–100; then the tier weights. A failed or invalid answer counts as wrong.

In [ ]:
records = [json.loads(line) for line in open(f"{run_dir}/results.jsonl")]


def competence(rs, kind, decisive_all):
    ts = [by_id[r["task_id"]] for r in rs]
    if kind == "choice":
        acc = statistics.mean(bool(r["correct"]) for r in rs)
        base = statistics.mean(1 / len(t.labels) for t in ts)
        return 100 * (acc - base) / (1 - base), None
    if kind == "noul":
        hits, decisive = [], []
        for r, t in zip(rs, ts):
            p = (r.get("probs") or {}).get("yes")
            sure = p is not None and (decisive_all or p >= NOUL_DECISIVE or p <= 1 - NOUL_DECISIVE)
            decisive.append(sure)
            hits.append(sure and bool(r["correct"]))
        return 100 * (statistics.mean(hits) - 0.5) / 0.5, statistics.mean(decisive)
    errs, base = [], []
    for r, t in zip(rs, ts):
        mid = (len(t.labels) - 1) / 2
        ev = r.get("ordinal_ev")
        errs.append(abs(ev - t.expected) if ev is not None else abs(len(t.labels) - 1))
        base.append(abs(mid - t.expected))
    if statistics.mean(base) == 0:
        base = [statistics.mean(abs(i - t.expected) for i in range(len(t.labels))) for t in ts]
    return 100 * (statistics.mean(base) - statistics.mean(errs)) / statistics.mean(base), None


def intelligence(decisive_all):
    rows, total = {}, 0.0
    for tier, weight in TIERS.items():
        per = {}
        for kind in ("choice", "noul", "score"):
            rs = [r for r in records if tier_of[r["task_id"]] == tier
                  and by_id[r["task_id"]].question["type"] == kind]
            if rs:
                per[kind] = competence(rs, kind, decisive_all)
        o1s = min(100.0, max(0.0, statistics.mean(c for c, _ in per.values())))
        rows[tier] = (per, o1s)
        total += weight * o1s
    return rows, total / sum(TIERS.values())


for label, decisive_all in (("as answered", False), ("noul made decisive", True)):
    rows, value = intelligence(decisive_all)
    print(f"\n{label}: Intelligence (public, tier-weighted) {value:.1f}")
    for tier, (per, o1s) in rows.items():
        parts = " | ".join(f"{k} {c:.1f}" + (f" (decisive {d:.0%})" if d is not None else "")
                           for k, (c, d) in per.items())
        print(f"  {tier:9s} O1S {o1s:5.1f} | {parts}")